# Phase 4: Individual Model Training Results

Verification notebook for Phase 4 of **Heterogeneous Graph Fusion for Multimodal Market Dynamics**.

All **3 individual models** have been trained with temporal train/val/test splits:
- **4A** Price CNN-BiLSTM
- **4B** Document FinBERT + AttentionPooling
- **4C** Macro State Model (MLP on macro regime indicators) — new in V2

> **V2 Note:** The original **4C News FinBERT+BiGRU** model was removed (files deleted; <2% training coverage). The original **4D Earnings Surprise MLP** was invalidated by data leakage (is_beat/is_miss features directly encoded the target label). In V2, the **4C** slot is occupied by the new **MacroStateModel** (MLP). Surprise features are retained as 5-dimensional backward-looking gating inputs in the fusion model.

In [ ]:
import json
import pathlib
import pandas as pd
import torch

ROOT = pathlib.Path.cwd().parent
results_path = ROOT / "models" / "phase4_results.json"

with open(results_path) as f:
    _raw = json.load(f)

# V2: filter to only price and document; skip news (removed: <2% coverage) and surprise (removed: data leakage)
models_to_show = {'price', 'document'}
results = {k: v for k, v in _raw.items() if k in models_to_show}

print(f"Results loaded from {results_path}")
print(f"Models: {list(results.keys())}")
print("News model (4C) and Surprise model (4D) removed in V2.")

## 1. Summary Table

In [ ]:
# V2: News (original 4C) and Surprise (4D) rows removed -- only Price (4A) and Document (4B) remain
rows = []
for name, r in results.items():
    tm = r["test_metrics"]
    rows.append({
        "Model": name.capitalize(),
        "Test Accuracy": f"{tm['accuracy']:.4f}",
        "Test F1": f"{tm['f1']:.4f}",
        "Test AUC": f"{tm['auc']:.4f}",
        "Majority Baseline": f"{r['baseline_accuracy']:.4f}",
        "Acc vs Baseline": f"{tm['accuracy'] - r['baseline_accuracy']:+.4f}",
        "Epochs": r.get("epochs_trained", "N/A"),
        "Time (s)": r.get("time_seconds", "N/A"),
    })

df = pd.DataFrame(rows)
df.set_index("Model", inplace=True)
print("Note: News (original 4C) and Surprise (4D) rows removed in V2. Table shows Price (4A) and Document (4B) only.")
df

## 2. Model Checkpoints Verification

In [ ]:
checkpoint_names = [
    "price_model_best.pt",
    "document_model_best.pt",
    "graph_model_best.pt",
]
# Note: news_model_best.pt and surprise_model_best.pt removed in V2

print("Checkpoint Verification")
print("=" * 60)
all_ok = True
for name in checkpoint_names:
    path = ROOT / "models" / name
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        ckpt = torch.load(path, weights_only=False, map_location="cpu")
        epoch = ckpt.get("epoch", "?")
        metrics = ckpt.get("metrics", {})
        print(f"  OK  {name} ({size_mb:.1f} MB, epoch {epoch})")
        if metrics:
            for k, v in metrics.items():
                if isinstance(v, float):
                    print(f"       {k}: {v:.4f}")
    else:
        print(f"  MISS  {name}")
        all_ok = False

print()
print("All checkpoints present:", "YES" if all_ok else "NO")

## 3. Performance Analysis

Individual modality models are expected to show modest performance since:
- Market direction is inherently hard to predict from any single signal
- The project hypothesis is that **graph fusion across modalities** will outperform individual models
- Small sample sizes (92 documents, 534 news samples) limit NLP model capacity

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# V2: only Price (4A) and Document (4B) are shown in these charts.
# News (original 4C, FinBERT+BiGRU) and Surprise (4D, MLP) were removed in V2 and are excluded from all bars.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

model_names = [name.capitalize() for name in results.keys()]
accs = [r["test_metrics"]["accuracy"] for r in results.values()]
f1s = [r["test_metrics"]["f1"] for r in results.values()]
aucs = [r["test_metrics"]["auc"] for r in results.values()]
baselines = [r["baseline_accuracy"] for r in results.values()]

x = np.arange(len(model_names))
width = 0.35

# Accuracy vs Baseline
ax = axes[0]
bars1 = ax.bar(x - width/2, accs, width, label="Test Accuracy", color="steelblue")
bars2 = ax.bar(x + width/2, baselines, width, label="Majority Baseline", color="lightcoral")
ax.set_ylabel("Accuracy")
ax.set_title("Test Accuracy vs Majority Baseline\n(Price 4A, Document 4B — V2)")
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15)
ax.legend()
ax.set_ylim(0, 1)

# F1 Scores
ax = axes[1]
ax.bar(model_names, f1s, color="seagreen")
ax.set_ylabel("F1 Score")
ax.set_title("Test F1 Score\n(News & Surprise removed in V2)")
ax.set_ylim(0, 1)
for i, v in enumerate(f1s):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)

# AUC Scores
ax = axes[2]
ax.bar(model_names, aucs, color="darkorange")
ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.7, label="Random (0.5)")
ax.set_ylabel("ROC AUC")
ax.set_title("Test ROC AUC")
ax.set_ylim(0, 1)
ax.legend()
for i, v in enumerate(aucs):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / "reports" / "phase4_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to reports/phase4_results.png")

## 4. Detailed Per-Model Metrics

In [ ]:
# V2: NEWS and SURPRISE detailed metrics blocks removed.
# - News FinBERT+BiGRU (original 4C): model files deleted; training coverage <2% of dataset
# - Surprise MLP (4D): invalidated due to data leakage (is_beat/is_miss features directly encoded the target)
# Only Price (4A) and Document (4B) detailed metrics are shown below.
print("Note: NEWS (original 4C) and SURPRISE (4D) metrics blocks removed in V2.")
print("  - News model (4C): files deleted, <2% training coverage")
print("  - Surprise model (4D): invalidated by data leakage")
print()

for name, r in results.items():
    tm = r["test_metrics"]
    bl = r["baseline_accuracy"]
    print(f"{'=' * 50}")
    print(f"  {name.upper()}")
    print(f"{'=' * 50}")
    print(f"  Epochs trained:     {r.get('epochs_trained', 'N/A')}")
    print(f"  Training time:      {r.get('time_seconds', 'N/A')}s")
    if 'n_train' in r:
        print(f"  Samples:            train={r['n_train']}, val={r['n_val']}, test={r['n_test']}")
    print(f"  Test accuracy:      {tm['accuracy']:.4f} (baseline: {bl:.4f}, delta: {tm['accuracy']-bl:+.4f})")
    print(f"  Test precision:     {tm['precision']:.4f}")
    print(f"  Test recall:        {tm['recall']:.4f}")
    print(f"  Test F1:            {tm['f1']:.4f}")
    print(f"  Test AUC:           {tm['auc']:.4f}")
    print()

## 5. Phase 4 Verification Checks

In [ ]:
checks = []

# Check 1: Core models trained (V2: price and document; MacroStateModel added separately as 4C)
expected = {"price", "document"}
check1 = set(results.keys()) == expected
checks.append(("All 2 core models trained (V2 adds MacroStateModel separately)", check1))

# Check 2: All have test metrics
check2 = all("test_metrics" in r for r in results.values())
checks.append(("All have test metrics", check2))

# Check 3: All checkpoints saved
check3 = all((ROOT / "models" / f"{n}_model_best.pt").exists() for n in results)
checks.append(("All checkpoints saved", check3))

# Check 4: All status OK
check4 = all(r.get("status") == "OK" for r in results.values())
checks.append(("All statuses OK", check4))

# Check 5: Metrics are valid
check5 = all(
    0 <= r["test_metrics"]["accuracy"] <= 1
    and 0 <= r["test_metrics"]["f1"] <= 1
    for r in results.values()
)
checks.append(("Metrics in valid range", check5))

# Check 6: No training failures
check6 = all(r.get("epochs_trained", 0) > 0 for r in results.values())
checks.append(("All models completed >0 epochs", check6))

print("Phase 4 Verification")
print("=" * 50)
all_pass = True
for desc, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}] {desc}")

print()
print(f"Overall: {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")

## Summary

Phase 4 individual model training is **complete**. Key findings:

1. **Price CNN-BiLSTM** (4A): Marginal signal above majority baseline (+0.49%). The model captures slight directional patterns in 60-day price windows.

2. **Document FinBERT** (4B): Below baseline (-6.7%). With only 52 training samples (one per ticker per year), the frozen FinBERT + head architecture lacks sufficient data to learn meaningful annual direction signals from 10-K filings.

3. **Macro State Model MLP** (4C — new in V2): Replaces the original 4C News FinBERT+BiGRU. The MacroStateModel is a lightweight MLP trained on macro regime indicators (yield curve slope, VIX regime, credit spreads, etc.) to produce a soft macro-state embedding used as a conditioning signal in the fusion model. It is trained and validated separately from the JSON-based results above, and its checkpoint is `macro_state_model_best.pt`.

> **V2 Removals:** The original **4C News FinBERT+BiGRU** (files deleted — training coverage <2%) and **4D Earnings Surprise MLP** (invalidated — data leakage via is_beat/is_miss labels) are no longer part of Phase 4. Surprise features are retained as 5-dimensional backward-looking inputs to the fusion model's gating mechanism.

These individual model results establish baselines for **Phase 5** (Graph-based feature fusion), where cross-modal interactions are expected to unlock additional signal.